# OANDA EURUSD Data Fetcher

Fetch EURUSD OHLCV data at 1-hour (H1) and 5-minute (M5) intervals for 2023

In [2]:
import v20
import pandas as pd
from datetime import datetime, timedelta
import time

## OANDA Account Setup

1. Create a practice account at https://www.oanda.com/demo-account/
2. Get your API token from the 'Manage API Access' section
3. We'll fetch the account ID automatically below

In [3]:
# OANDA API Configuration
API_URL = 'api-fxpractice.oanda.com'
ACCESS_TOKEN = 'c3cb7c6a198247f404843c4878d4295b-ce3f4461613aab8832675ea50762c281'  # Replace with your token

# Initialize API connection
api = v20.Context(
    API_URL,
    443,
    token=ACCESS_TOKEN
)

# Get account ID automatically
try:
    response = api.account.list()
    if response.status != 200:
        raise Exception(f'Error getting account list: {response.status}')
        
    accounts = response.body['accounts']
    if not accounts:
        raise Exception('No accounts found')
        
    ACCOUNT_ID = accounts[0].id  # Use the first available account
    print(f'Successfully retrieved account ID: {ACCOUNT_ID}')
except Exception as e:
    print(f'Error: {e}')
    print('Please check your API token and connection')

Successfully retrieved account ID: 101-004-31500681-001


In [6]:
def fetch_candles(instrument, granularity, start_date, end_date):
    candles = []
    current_date = start_date
    
    while current_date < end_date:
        next_date = min(current_date + timedelta(days=5), end_date)
        
        params = {
            'from': current_date.isoformat('T') + 'Z',
            'to': next_date.isoformat('T') + 'Z',
            'granularity': granularity,
            'price': 'MBA'  # Bid/Mid/Ask
        }
        
        try:
            response = api.instrument.candles(instrument, **params)
            # Access the candles directly from response.body
            if hasattr(response, 'body') and 'candles' in response.body:
                candles.extend(response.body['candles'])
            else:
                print(f'No candles in response for {current_date}')
        except Exception as e:
            print(f'Error fetching data: {str(e)}')
            break
            
        current_date = next_date
        time.sleep(0.5)  # Rate limiting
    
    return candles

In [7]:
# Set date range
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)

# Fetch H1 and M5 data
h1_candles = fetch_candles('EUR_USD', 'H1', start_date, end_date)
m5_candles = fetch_candles('EUR_USD', 'M5', start_date, end_date)

print(f'Fetched {len(h1_candles)} H1 candles')
print(f'Fetched {len(m5_candles)} M5 candles')

Fetched 36500 H1 candles
Fetched 36500 M5 candles


In [8]:
def process_candles(candles):
    data = []
    for candle in candles:
        if candle.complete:
            data.append({
                'datetime': candle.time,
                'open': float(candle.mid.o),
                'high': float(candle.mid.h),
                'low': float(candle.mid.l),
                'close': float(candle.mid.c),
                'volume': int(candle.volume)
            })
    return pd.DataFrame(data)

# Convert to DataFrames and save
h1_df = process_candles(h1_candles)
m5_df = process_candles(m5_candles)

h1_df.to_csv('H1.csv', index=False)
m5_df.to_csv('M5.csv', index=False)

In [9]:
# Display sample of H1 data
print("H1 Data Sample:")
display(h1_df.head())

print("\nM5 Data Sample:")
display(m5_df.head())

H1 Data Sample:


,datetime,open,high,low,close,volume
0,2025-03-24T21:00:00.000000000Z,1.08013,1.08019,1.07973,1.08006,739
1,2025-03-24T22:00:00.000000000Z,1.08006,1.08030,1.07952,1.08026,926
2,2025-03-24T23:00:00.000000000Z,1.08027,1.08038,1.08006,1.08026,1056
3,2025-03-25T00:00:00.000000000Z,1.08025,1.08030,1.07962,1.08002,2514
4,2025-03-25T01:00:00.000000000Z,1.08002,1.08087,1.08000,1.08022,3776



M5 Data Sample:


,datetime,open,high,low,close,volume
0,2025-04-20T22:35:00.000000000Z,1.14273,1.14274,1.14198,1.14250,533
1,2025-04-20T22:40:00.000000000Z,1.14248,1.14301,1.14182,1.14298,552
2,2025-04-20T22:45:00.000000000Z,1.14298,1.14324,1.14296,1.14313,411
3,2025-04-20T22:50:00.000000000Z,1.14312,1.14367,1.14308,1.14358,284
4,2025-04-20T22:55:00.000000000Z,1.14360,1.14372,1.14257,1.14332,548


In [2]:
import pandas as pd
import pandas_ta as ta
import matplotlib.pyplot as plt
import numpy as np

# Load the data
h1_data = pd.read_csv('H1_cleaned.csv')
m5_data = pd.read_csv('M5_cleaned.csv')

# Convert datetime columns if needed
if 'datetime' in h1_data.columns or 'time' in h1_data.columns:
    datetime_col = 'datetime' if 'datetime' in h1_data.columns else 'time'
    h1_data[datetime_col] = pd.to_datetime(h1_data[datetime_col])
    h1_data.set_index(datetime_col, inplace=True)

if 'datetime' in m5_data.columns or 'time' in m5_data.columns:
    datetime_col = 'datetime' if 'datetime' in m5_data.columns else 'time'
    m5_data[datetime_col] = pd.to_datetime(m5_data[datetime_col])
    m5_data.set_index(datetime_col, inplace=True)

# Check and rename columns if necessary
# OANDA typically uses 'c' for close, 'o' for open, etc.
column_mapping = {
    'c': 'close', 'o': 'open', 'h': 'high', 'l': 'low', 'v': 'volume'
}

for old, new in column_mapping.items():
    if old in h1_data.columns and new not in h1_data.columns:
        h1_data[new] = h1_data[old]
    if old in m5_data.columns and new not in m5_data.columns:
        m5_data[new] = m5_data[old]

# Calculate 1H indicators
print("Calculating 1H indicators...")
# Trend Bias: 8-period EMA on 1-hour chart
h1_data['ema_8'] = ta.ema(h1_data['close'], length=8)
h1_data['trend_bias'] = np.where(h1_data['close'] > h1_data['ema_8'], 'bullish', 'bearish')

# Calculate 5M indicators
print("Calculating 5M indicators...")
# Entry Points: 5-period EMA on 5-minute chart
m5_data['ema_5'] = ta.ema(m5_data['close'], length=5)
m5_data['price_crossed_above_ema'] = np.where(
    (m5_data['close'] > m5_data['ema_5']) & (m5_data['close'].shift(1) <= m5_data['ema_5'].shift(1)), 
    True, False
)
m5_data['price_crossed_below_ema'] = np.where(
    (m5_data['close'] < m5_data['ema_5']) & (m5_data['close'].shift(1) >= m5_data['ema_5'].shift(1)), 
    True, False
)

# RSI indicator for divergence (alternative entry)
m5_data['rsi_9'] = ta.rsi(m5_data['close'], length=9)

# Exit trades: MACD (8, 17, 9) on 5-minute chart
macd = ta.macd(m5_data['close'], fast=8, slow=17, signal=9)
m5_data = m5_data.join(macd)

# Identify MACD crossovers
m5_data['macd_cross_below'] = np.where(
    (m5_data['MACD_8_17_9'] < m5_data['MACDs_8_17_9']) & 
    (m5_data['MACD_8_17_9'].shift(1) >= m5_data['MACDs_8_17_9'].shift(1)), 
    True, False
)
m5_data['macd_cross_above'] = np.where(
    (m5_data['MACD_8_17_9'] > m5_data['MACDs_8_17_9']) & 
    (m5_data['MACD_8_17_9'].shift(1) <= m5_data['MACDs_8_17_9'].shift(1)), 
    True, False
)

# Save the results
h1_data.to_csv('H1_with_indicators.csv')
m5_data.to_csv('M5_with_indicators.csv')

print("1H Data Sample:")
print(h1_data[['close', 'ema_8', 'trend_bias']].tail())

print("\n5M Data Sample:")
print(m5_data[['close', 'ema_5', 'rsi_9', 'MACD_8_17_9', 'MACDs_8_17_9', 
               'macd_cross_above', 'macd_cross_below']].tail())

# Function to visualize the indicators for a specific time period (optional)
def visualize_indicators(h1_sample, m5_sample, period=50):
    """
    Visualize the indicators for analysis
    """
    # 1H Chart with EMA
    plt.figure(figsize=(14, 7))
    plt.subplot(2, 1, 1)
    plt.title('1H Chart - Trend Bias')
    plt.plot(h1_sample.index[-period:], h1_sample['close'][-period:], label='Close')
    plt.plot(h1_sample.index[-period:], h1_sample['ema_8'][-period:], label='EMA(8)')
    plt.legend()
    plt.grid(True)
    
    # 5M Chart with Entry/Exit signals
    plt.subplot(2, 1, 2)
    plt.title('5M Chart - Entry & Exit Points')
    plt.plot(m5_sample.index[-period*12:], m5_sample['close'][-period*12:], label='Close')
    plt.plot(m5_sample.index[-period*12:], m5_sample['ema_5'][-period*12:], label='EMA(5)')
    
    # Plot entry signals
    entries_long = m5_sample[-period*12:][m5_sample['price_crossed_above_ema'][-period*12:]]
    entries_short = m5_sample[-period*12:][m5_sample['price_crossed_below_ema'][-period*12:]]
    plt.scatter(entries_long.index, entries_long['close'], color='green', marker='^', s=100, label='Long Entry')
    plt.scatter(entries_short.index, entries_short['close'], color='red', marker='v', s=100, label='Short Entry')
    
    # Plot exit signals
    exits_long = m5_sample[-period*12:][m5_sample['macd_cross_below'][-period*12:]]
    exits_short = m5_sample[-period*12:][m5_sample['macd_cross_above'][-period*12:]]
    plt.scatter(exits_long.index, exits_long['close'], color='orange', marker='x', s=100, label='Long Exit')
    plt.scatter(exits_short.index, exits_short['close'], color='purple', marker='x', s=100, label='Short Exit')
    
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('indicator_analysis.png')
    plt.show()

# Uncomment to visualize (if you want to see the indicators graphically)
# visualize_indicators(h1_data, m5_data)

print("Indicators calculated and saved to CSV files!")

Calculating 1H indicators...
Calculating 5M indicators...
1H Data Sample:
                             close     ema_8 trend_bias
time                                                   
2025-04-25 16:00:00+00:00  1.13687  1.136215    bullish
2025-04-25 17:00:00+00:00  1.13765  1.136534    bullish
2025-04-25 18:00:00+00:00  1.13794  1.136847    bullish
2025-04-25 19:00:00+00:00  1.13592  1.136641    bearish
2025-04-25 20:00:00+00:00  1.13628  1.136560    bearish

5M Data Sample:
                             close     ema_5      rsi_9  MACD_8_17_9  \
time                                                                   
2025-04-25 20:35:00+00:00  1.13618  1.135996  40.529735    -0.000394   
2025-04-25 20:40:00+00:00  1.13606  1.136017  37.857560    -0.000356   
2025-04-25 20:45:00+00:00  1.13578  1.135938  32.272224    -0.000351   
2025-04-25 20:50:00+00:00  1.13597  1.135949  39.128081    -0.000319   
2025-04-25 20:55:00+00:00  1.13628  1.136059  48.666131    -0.000254   

            